# 02 — Pandas Fundamentals

**Topics covered:**
1. Series & DataFrame basics
2. Loading & inspecting data
3. Selecting & filtering
4. Cleaning: missing values & duplicates
5. Adding & transforming columns
6. GroupBy & aggregation
7. Merging DataFrames
8. Time series
9. Mini-exercises

In [ ]:
import numpy as np
import pandas as pd
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from ds_sandbox.dataset import load_iris, make_sales, make_timeseries

print(f'pandas version: {pd.__version__}')

## 1. Series & DataFrame Basics

In [ ]:
# Series
s = pd.Series([10, 20, 30, 40], index=['a', 'b', 'c', 'd'], name='scores')
print(s)
print('\nMean:', s.mean(), '| Max:', s.max())

In [ ]:
# DataFrame from dict
df = pd.DataFrame({
    'name':  ['Alice', 'Bob', 'Carol', 'Dave'],
    'age':   [25, 30, 35, 28],
    'score': [88.5, 92.0, 79.3, 95.1],
    'passed': [True, True, False, True],
})
df

## 2. Loading & Inspecting Data

In [ ]:
iris = load_iris()

print('Shape       :', iris.shape)
print('Columns     :', iris.columns.tolist())
print('\nData types:\n', iris.dtypes)
print('\nMissing values:\n', iris.isnull().sum())
iris.head()

In [ ]:
iris.describe()

## 3. Selecting & Filtering

In [ ]:
# Column selection
print(iris[['sepal length (cm)', 'species']].head())

# Boolean filter
setosa = iris[iris['species'] == 'setosa']
print(f'\nSetosa rows: {len(setosa)}')

# .loc (label) and .iloc (position)
print('\n.loc [0:2, name+species]:')
print(iris.loc[0:2, ['sepal length (cm)', 'species']])

print('\n.iloc [0:3, 0:2]:')
print(iris.iloc[0:3, 0:2])

## 4. Cleaning: Missing Values & Duplicates

In [ ]:
# Inject some NaN values for practice
dirty = iris.copy()
rng = np.random.default_rng(0)
idx = rng.choice(len(dirty), size=10, replace=False)
dirty.loc[idx, 'sepal length (cm)'] = np.nan

print('Missing before:', dirty.isnull().sum().sum())

# Option A: drop rows with any NaN
dropped = dirty.dropna()
print('Rows after dropna:', len(dropped))

# Option B: fill with column mean
filled = dirty.fillna(dirty.mean(numeric_only=True))
print('Missing after fillna:', filled.isnull().sum().sum())

# Duplicates
print('Duplicate rows:', iris.duplicated().sum())

## 5. Adding & Transforming Columns

In [ ]:
df = iris.copy()

# New column from arithmetic
df['sepal_area'] = df['sepal length (cm)'] * df['sepal width (cm)']

# Apply / map
df['length_category'] = pd.cut(
    df['sepal length (cm)'], bins=3, labels=['small', 'medium', 'large']
)

# Rename columns (clean up spaces)
df = df.rename(columns=lambda c: c.replace(' ', '_').replace('(cm)', '').strip('_'))

df[['sepal_length', 'sepal_area', 'length_category', 'species']].head(10)

## 6. GroupBy & Aggregation

In [ ]:
sales = make_sales()

# Revenue by region
region_rev = sales.groupby('region')['revenue'].sum().sort_values(ascending=False)
print('Revenue by region:\n', region_rev)

# Multi-level aggregation
summary = sales.groupby(['region', 'product']).agg(
    total_units=('units_sold', 'sum'),
    total_revenue=('revenue', 'sum'),
    avg_price=('unit_price', 'mean'),
).round(2)

print('\nSummary table:')
summary.head(8)

In [ ]:
# Pivot table
pivot = sales.pivot_table(
    values='revenue', index='product', columns='region', aggfunc='sum'
).round(0)
pivot

## 7. Merging DataFrames

In [ ]:
products = pd.DataFrame({
    'product': ['Widget A', 'Widget B', 'Gadget X', 'Gadget Y'],
    'category': ['Widget', 'Widget', 'Gadget', 'Gadget'],
    'weight_kg': [0.5, 0.8, 1.2, 2.0],
})

# Inner join (only matching keys)
merged = pd.merge(sales, products, on='product', how='left')
print('Merged shape:', merged.shape)
merged[['product', 'category', 'revenue', 'weight_kg']].head()

## 8. Time Series

In [ ]:
ts = make_timeseries()
ts = ts.set_index('date')

print('Date range:', ts.index.min(), '->', ts.index.max())

# Resample to monthly mean
monthly = ts.resample('ME').mean()
print('\nMonthly mean (first 6):')
print(monthly.head(6))

# Rolling 7-day average
ts['rolling_7'] = ts['value'].rolling(7).mean()
ts.head(10)

## 9. Mini-Exercises

In [ ]:
# Exercise 1: From the iris dataset, find the species with the highest mean petal length
# Your code here


In [ ]:
# Exercise 2: From the sales dataset, find the top 3 (region, product) combos by revenue
# Your code here


In [ ]:
# Exercise 3: Compute a 30-day rolling standard deviation on the timeseries
# Your code here
